In [1]:
! python /home/bsb2144/daart/examples/fit_models_subsample_loop.py --pre wv2 --fit_tcn --frac 1 --dataset ibl --n_samples 1 --data /home/bsb2144/daart_utils/configs/data_ibl2.yaml --model /home/bsb2144/daart_utils/configs/model_ibl2.yaml --train /home/bsb2144/daart_utils/configs/train_ibl2.yaml


here
mod types ['dtcn']
frac 1
n_sessions 18
['dc962048-89bb-4e6a-96a9-b062a2be1426', '1b715600-0cbc-442c-bd00-5b0ac2865de1', '8928f98a-b411-497e-aa4b-aa752434686d', 'd0ea3148-948d-4817-94f8-dcaf2342bbbe', 'e535fb62-e245-4a48-b119-88ce62a6fe67', '3bcb81b4-d9ca-4fc9-a1cd-353a966239ca', '54238fd6-d2d0-4408-b1a9-d19d24fd29ce', 'f140a2ec-fd49-4814-994a-fe3476f14e66', '73918ae1-e4fd-4c18-b132-00cb555b1ad2', '6f09ba7e-e3ce-44b0-932b-c003fb44fb89', '7cb81727-2097-4b52-b480-c89867b5b34c', 'db4df448-e449-4a6f-a0e7-288711e7a75a', 'ca4ecb4c-4b60-4723-9b9e-2c54a6290a53', 'b22f694e-4a34-4142-ab9d-2556c3487086', 'f115196e-8dfe-4d2a-8af3-8206d93c1729', 'ff96bfe1-d925-4553-94b5-bf8297adf259', 'b03fbc44-3d8e-4a6c-8a50-5ea3498568e0', '3638d102-e8b6-4230-8742-e548cd87a949']
['dc962048-89bb-4e6a-96a9-b062a2be1426', '1b715600-0cbc-442c-bd00-5b0ac2865de1', '8928f98a-b411-497e-aa4b-aa752434686d', 'd0ea3148-948d-4817-94f8-dcaf2342bbbe', 'e535fb62-e245-4a48-b119-88ce62a6fe67', '3bcb81b4-d9ca-4fc9-a1cd-353a9662

In [4]:

import yaml
import os
import subprocess
import pandas as pd
import numpy as np
from itertools import product
import glob

def create_param_string(model_params):
    param_strings = []
    for key, val in model_params.items():
        param_strings.append(f"{key}={val}")
    return "_".join(param_strings)

def update_yaml_files_and_save_params(train_yaml_file, model_yaml_file, lr_value, model_params):
    """
    Update two YAML files with specific parameters and return a parameter info string
    """
    # Load and update the train YAML file
    with open(train_yaml_file, 'r') as file:
        train_data = yaml.safe_load(file)
    
    # Update learning rate in train config
    train_data['learning_rate'] = lr_value
    
    # Save the updated train YAML
    with open(train_yaml_file, 'w') as file:
        yaml.dump(train_data, file, default_flow_style=False)
    
    # Load and update the model YAML file
    with open(model_yaml_file, 'r') as file:
        model_data = yaml.safe_load(file)
    
    # Update model parameters
    for key, value in model_params.items():
        model_data[key] = value
    
    # Save the updated model YAML
    with open(model_yaml_file, 'w') as file:
        yaml.dump(model_data, file, default_flow_style=False)
    
    # Create a parameter info string combining all parameters
    param_info = f"lr={lr_value}_" + create_param_string(model_params)
    
    return param_info

def run_training_script(data_config, model_config, train_config, dataset, param_info):
    """
    Run the training script with the specified arguments
    """
    if vel:
        new_pre = "vel_" + param_info
    else:
        new_pre = param_info
    cmd = [
        "python", "/home/bsb2144/daart/examples/fit_models_subsample_loop.py",
        "--pre", "t9_"+new_pre,  # Use param_info as the --pre argument
        "--fit_tcn",
        "--frac", "1",
        "--dataset", dataset,
        "--n_samples", "1",
        "--data", data_config,
        "--model", model_config,
        "--train", train_config
    ]
    
    print(f"Running: {' '.join(cmd)}")
    print(f"With parameters: {param_info}")
    
    # Use subprocess.run to execute the command
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error running script: {result.stderr}")
        return False
    else:
        print("Script executed successfully")
        return True

def evaluate_directory(dir_name, n_vs):
    """
    Evaluate a directory by finding the minimum val_loss where dataset=-1 in each version
    and averaging them.
    
    Parameters:
    dir_name (str): Directory name to evaluate
    
    Returns:
    float: Average of minimum val_loss values across versions
    """
    min_val_losses = []
    
    # Process each version subdirectory
    for version in range(n_vs):  # version_0 through version_4
        version_dir = os.path.join(dir_name, f"version_{version}")
        metrics_path = os.path.join(version_dir, "metrics.csv")
        
        try:
            # Read the metrics CSV file
            df = pd.read_csv(metrics_path)
            
            # Filter rows where dataset=-1 and find minimum val_loss
            filtered_df = df[df['dataset'] == -1]
            if not filtered_df.empty:
                min_val_loss = filtered_df['val_loss'].min()
                min_val_losses.append(min_val_loss)
                #print(f"Min val_loss for {version_dir}: {min_val_loss}")
        
        except Exception as e:
            print(f"Error processing {metrics_path}: {e}")
    
    # Calculate average if we have any valid values
    if min_val_losses:
        avg_score = np.mean(min_val_losses)
        print(f"Average min val_loss for {dir_name}: {avg_score}")
        return avg_score
    else:
        print(f"No valid data found for {dir_name}")
        return float('inf')  # Return infinity if no valid data

def get_hyperparameters(dir_name):
    """
    Get hyperparameters from the version_0/hparams.yaml file
    
    Parameters:
    dir_name (str): Directory name
    
    Returns:
    dict: Hyperparameters from the YAML file
    """
    hparams_path = os.path.join(dir_name, "version_0", "hparams.yaml")
    
    try:
        with open(hparams_path, 'r') as file:
            hparams = yaml.safe_load(file)
            return hparams
    except Exception as e:
        print(f"Error reading {hparams_path}: {e}")
        return None

def grid_search_hyperparameters(train_yaml_file, model_yaml_file, data_config, model_config, train_config, dataset, base_dir, n_vs):
    """
    Perform grid search over hyperparameter combinations, run training script, and evaluate results
    """
    # Define hyperparameter values to search
      
    # lr=1e-05_dropout=0.1_n_hid_units=32_n_lags=16
    # learning_rates = [0.0001, 0.00001]#, 0.001]
    # dropout_rates = [0.1]
    # hidden_units = [32, 64]
    # n_lags = [4, 8, 16]
    # n_hid_layers = [2]


    learning_rates = [0.001]
    dropout_rates = [0.1]
    hidden_units = [96]
    n_lags = [4]
    n_hid_layers = [2]
    
    # Store all parameter info strings
    param_info_list = []
    
    # Loop through all combinations
    for lr, drop, hidden,lag, n_layer in product(learning_rates, dropout_rates, hidden_units, n_lags, n_hid_layers):
        # Create model parameters dictionary
        model_params = {
            'dropout': drop,
            'n_hid_units': hidden,
            "n_lags": lag,
            "n_hid_layers": n_layer
        }
        
        # Update YAML files and get parameter info
        param_info = update_yaml_files_and_save_params(
            train_yaml_file, 
            model_yaml_file, 
            lr, 
            model_params
        )
        param_info_list.append(param_info)
        
        # Run the training script with updated configs
        success = run_training_script(data_config, model_config, train_config, dataset, param_info)
        if not success:
            print(f"Skipping evaluation for {param_info} due to training failure")
    
    print(f"Completed {len(param_info_list)} parameter combinations")
    
    # Evaluate directories after all runs complete
    evaluate_and_find_best_model(param_info_list, base_dir, n_vs)

def evaluate_and_find_best_model(param_info_list, base_dir, n_vs):
    """
    Evaluate all directories and find the one with the smallest average val_loss
    
    Parameters:
    param_info_list (list): List of parameter info strings (directory names)
    """
    # Dictionary to store directory scores
    scores = {}
    # Evaluate each directory
    
    for param_info in param_info_list:
        if vel:
            new_pre = "vel_" + param_info
        else:
            new_pre = param_info
        score = evaluate_directory(base_dir.format(new_pre), n_vs)
        scores[param_info] = score
    
    # Find directory with the smallest score
    if scores:
        best_dir = min(scores, key=scores.get)
        best_score = scores[best_dir]
        
        print("\n" + "="*50)
        print(f"Best directory: {best_dir}")
        print(f"Best average val_loss: {best_score}")
        
        return best_dir, best_score
    else:
        print("No valid directories to evaluate")
        return None, None


top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved


  0%|                                                   | 0/502 [00:00<?, ?it/s]/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/torch/nn/functional.py:1338: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  warnings.warn("dropout2d: Received a 3D input to dropout2d and assuming that channel-wise "
100%|███████████████████████████████████████| 502/502 [2:09:28<00:00, 15.48s/it]
The for loop took 7768.620015 seconds to complete.
The for loop took 7768.620015 seconds to complete.
   dataset  dtype  epoch                 loss         val
0      all  train   11.0                 loss  228.362358
0      all  train   11.0  

findfont: Matching sans\-serif:style=normal:variant=normal:weight=normal:stretch=normal:size=16.5.
findfont: score(FontEntry(fname='/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/matplotlib/mpl-data/fonts/ttf/cmss10.ttf', name='cmss10', style='normal', variant='normal', weight=400, stretch='normal', size='scalable')) = 10.05
findfont: score(FontEntry(fname='/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/matplotlib/mpl-data/fonts/ttf/STIXSizTwoSymReg.ttf', name='STIXSizeTwoSym', style='normal', variant='normal', weight=400, stretch='normal', size='scalable')) = 10.05
findfont: score(FontEntry(fname='/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/matplotlib/mpl-data/fonts/ttf/DejaVuSerifDisplay.ttf', name='DejaVu Serif Display', style='normal', variant='normal', weight=400, stretch='normal', size='scalable')) = 10.05
findfont: score(FontEntry(fname='/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/matplotlib/mpl-data/f

findfont: Matching sans\-serif:style=normal:variant=normal:weight=normal:stretch=normal:size=18.0.
findfont: score(FontEntry(fname='/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/matplotlib/mpl-data/fonts/ttf/cmss10.ttf', name='cmss10', style='normal', variant='normal', weight=400, stretch='normal', size='scalable')) = 10.05
findfont: score(FontEntry(fname='/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/matplotlib/mpl-data/fonts/ttf/STIXSizTwoSymReg.ttf', name='STIXSizeTwoSym', style='normal', variant='normal', weight=400, stretch='normal', size='scalable')) = 10.05
findfont: score(FontEntry(fname='/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/matplotlib/mpl-data/fonts/ttf/DejaVuSerifDisplay.ttf', name='DejaVu Serif Display', style='normal', variant='normal', weight=400, stretch='normal', size='scalable')) = 10.05
findfont: score(FontEntry(fname='/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/matplotlib/mpl-data/f

top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved


...

top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved


  0%|                                                   | 0/502 [00:00<?, ?it/s]/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/torch/nn/functional.py:1338: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  warnings.warn("dropout2d: Received a 3D input to dropout2d and assuming that channel-wise "
100%|███████████████████████████████████████| 502/502 [2:09:35<00:00, 15.49s/it]
The for loop took 7775.009167 seconds to complete.
The for loop took 7775.009167 seconds to complete.
   dataset  dtype  epoch                 loss         val
0      all  train   11.0                 loss  238.049902
0      all  train   11.0  

findfont: Matching sans\-serif:style=normal:variant=normal:weight=normal:stretch=normal:size=16.5.
findfont: score(FontEntry(fname='/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/matplotlib/mpl-data/fonts/ttf/cmss10.ttf', name='cmss10', style='normal', variant='normal', weight=400, stretch='normal', size='scalable')) = 10.05
findfont: score(FontEntry(fname='/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/matplotlib/mpl-data/fonts/ttf/STIXSizTwoSymReg.ttf', name='STIXSizeTwoSym', style='normal', variant='normal', weight=400, stretch='normal', size='scalable')) = 10.05
findfont: score(FontEntry(fname='/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/matplotlib/mpl-data/fonts/ttf/DejaVuSerifDisplay.ttf', name='DejaVu Serif Display', style='normal', variant='normal', weight=400, stretch='normal', size='scalable')) = 10.05
findfont: score(FontEntry(fname='/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/matplotlib/mpl-data/f

findfont: Matching sans\-serif:style=normal:variant=normal:weight=normal:stretch=normal:size=18.0.
findfont: score(FontEntry(fname='/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/matplotlib/mpl-data/fonts/ttf/cmss10.ttf', name='cmss10', style='normal', variant='normal', weight=400, stretch='normal', size='scalable')) = 10.05
findfont: score(FontEntry(fname='/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/matplotlib/mpl-data/fonts/ttf/STIXSizTwoSymReg.ttf', name='STIXSizeTwoSym', style='normal', variant='normal', weight=400, stretch='normal', size='scalable')) = 10.05
findfont: score(FontEntry(fname='/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/matplotlib/mpl-data/fonts/ttf/DejaVuSerifDisplay.ttf', name='DejaVu Serif Display', style='normal', variant='normal', weight=400, stretch='normal', size='scalable')) = 10.05
findfont: score(FontEntry(fname='/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/matplotlib/mpl-data/f

top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved
top of Axes not in the figure, so title not moved


In [5]:
# Configuration files
data_config = "/home/bsb2144/daart_utils/configs/data_ibl3.yaml"
model_config = "/home/bsb2144/daart_utils/configs/model_ibl3.yaml"
train_config = "/home/bsb2144/daart_utils/configs/train_ibl3.yaml"
vel = False
# base model dir
base_dir = "/home/bsb2144/daart/results_daart/ibl/multi-22/dtcn/{}-18-good_sample-0_markers_100_np"
#base_dir = "/home/bsb2144/daart/results_daart/calms21/multi-0/dtcn/{}-68-good_sample-0_features-simba"
# Dataset name
dataset = "ibl"

# number of trial versions
n_vs=3


# YAML files to update with hyperparameters
train_yaml_file = train_config  # Contains learning_rate
model_yaml_file = model_config  # Contains dropout and n_hidden_units

grid_search_hyperparameters(
    train_yaml_file, 
    model_yaml_file, 
    data_config, 
    model_config, 
    train_config, 
    dataset,
    base_dir,
    n_vs
)

Running: python /home/bsb2144/daart/examples/fit_models_subsample_loop.py --pre t9_lr=0.001_dropout=0.1_n_hid_units=96_n_lags=4_n_hid_layers=2 --fit_tcn --frac 1 --dataset ibl --n_samples 1 --data /home/bsb2144/daart_utils/configs/data_ibl3.yaml --model /home/bsb2144/daart_utils/configs/model_ibl3.yaml --train /home/bsb2144/daart_utils/configs/train_ibl3.yaml
With parameters: lr=0.001_dropout=0.1_n_hid_units=96_n_lags=4_n_hid_layers=2
Script executed successfully
Completed 1 parameter combinations
Error processing /home/bsb2144/daart/results_daart/ibl/multi-22/dtcn/lr=0.001_dropout=0.1_n_hid_units=96_n_lags=4_n_hid_layers=2-18-good_sample-0_markers_100_np/version_0/metrics.csv: [Errno 2] No such file or directory: '/home/bsb2144/daart/results_daart/ibl/multi-22/dtcn/lr=0.001_dropout=0.1_n_hid_units=96_n_lags=4_n_hid_layers=2-18-good_sample-0_markers_100_np/version_0/metrics.csv'
Error processing /home/bsb2144/daart/results_daart/ibl/multi-22/dtcn/lr=0.001_dropout=0.1_n_hid_units=96_n_l

In [ ]:
!  python /home/bsb2144/daart/examples/fit_models_subsample_loop.py --pre t13_lr=0.001_dropout=0.1_n_hid_units=96_n_lags=4_n_hid_layers=2 --fit_tcn --frac 1 --dataset ibl --n_samples 1 --data /home/bsb2144/daart_utils/configs/data_ibl3.yaml --model /home/bsb2144/daart_utils/configs/model_ibl3.yaml --train /home/bsb2144/daart_utils/configs/train_ibl3.yaml

here
mod types ['dtcn']
frac 1
n_sessions 18
['dc962048-89bb-4e6a-96a9-b062a2be1426', '1b715600-0cbc-442c-bd00-5b0ac2865de1', '8928f98a-b411-497e-aa4b-aa752434686d', 'd0ea3148-948d-4817-94f8-dcaf2342bbbe', 'e535fb62-e245-4a48-b119-88ce62a6fe67', '3bcb81b4-d9ca-4fc9-a1cd-353a966239ca', '54238fd6-d2d0-4408-b1a9-d19d24fd29ce', 'f140a2ec-fd49-4814-994a-fe3476f14e66', '73918ae1-e4fd-4c18-b132-00cb555b1ad2', '6f09ba7e-e3ce-44b0-932b-c003fb44fb89', '7cb81727-2097-4b52-b480-c89867b5b34c', 'db4df448-e449-4a6f-a0e7-288711e7a75a', 'ca4ecb4c-4b60-4723-9b9e-2c54a6290a53', 'b22f694e-4a34-4142-ab9d-2556c3487086', 'f115196e-8dfe-4d2a-8af3-8206d93c1729', 'ff96bfe1-d925-4553-94b5-bf8297adf259', 'b03fbc44-3d8e-4a6c-8a50-5ea3498568e0', '3638d102-e8b6-4230-8742-e548cd87a949']
['dc962048-89bb-4e6a-96a9-b062a2be1426', '1b715600-0cbc-442c-bd00-5b0ac2865de1', '8928f98a-b411-497e-aa4b-aa752434686d', 'd0ea3148-948d-4817-94f8-dcaf2342bbbe', 'e535fb62-e245-4a48-b119-88ce62a6fe67', '3bcb81b4-d9ca-4fc9-a1cd-353a9662

In [2]:
! python /home/bsb2144/daart/examples/fit_models_subsample_loop.py --pre wvp5 --fit_tcn --frac 1 --dataset ibl --n_samples 1 --data /home/bsb2144/daart_utils/configs/data_ibl5.yaml --model /home/bsb2144/daart_utils/configs/model_ibl2.yaml --train /home/bsb2144/daart_utils/configs/train_ibl2.yaml


here
mod types ['dtcn']
frac 1
n_sessions 18
['dc962048-89bb-4e6a-96a9-b062a2be1426', '1b715600-0cbc-442c-bd00-5b0ac2865de1', '8928f98a-b411-497e-aa4b-aa752434686d', 'd0ea3148-948d-4817-94f8-dcaf2342bbbe', 'e535fb62-e245-4a48-b119-88ce62a6fe67', '3bcb81b4-d9ca-4fc9-a1cd-353a966239ca', '54238fd6-d2d0-4408-b1a9-d19d24fd29ce', 'f140a2ec-fd49-4814-994a-fe3476f14e66', '73918ae1-e4fd-4c18-b132-00cb555b1ad2', '6f09ba7e-e3ce-44b0-932b-c003fb44fb89', '7cb81727-2097-4b52-b480-c89867b5b34c', 'db4df448-e449-4a6f-a0e7-288711e7a75a', 'ca4ecb4c-4b60-4723-9b9e-2c54a6290a53', 'b22f694e-4a34-4142-ab9d-2556c3487086', 'f115196e-8dfe-4d2a-8af3-8206d93c1729', 'ff96bfe1-d925-4553-94b5-bf8297adf259', 'b03fbc44-3d8e-4a6c-8a50-5ea3498568e0', '3638d102-e8b6-4230-8742-e548cd87a949']
['dc962048-89bb-4e6a-96a9-b062a2be1426', '1b715600-0cbc-442c-bd00-5b0ac2865de1', '8928f98a-b411-497e-aa4b-aa752434686d', 'd0ea3148-948d-4817-94f8-dcaf2342bbbe', 'e535fb62-e245-4a48-b119-88ce62a6fe67', '3bcb81b4-d9ca-4fc9-a1cd-353a9662

In [ ]:
# try w all data for wv3 vit-cm_new

In [22]:
! python /home/bsb2144/daart/examples/fit_models_subsample_loop.py --pre wv4 --fit_tcn --frac 1 --dataset ibl --n_samples 1 --data /home/bsb2144/daart_utils/configs/data_ibl.yaml --model /home/bsb2144/daart_utils/configs/model_ibl2.yaml --train /home/bsb2144/daart_utils/configs/train_ibl2.yaml


here
mod types ['dtcn']
frac 1
n_sessions 18
['dc962048-89bb-4e6a-96a9-b062a2be1426', '1b715600-0cbc-442c-bd00-5b0ac2865de1', '8928f98a-b411-497e-aa4b-aa752434686d', 'd0ea3148-948d-4817-94f8-dcaf2342bbbe', 'e535fb62-e245-4a48-b119-88ce62a6fe67', '3bcb81b4-d9ca-4fc9-a1cd-353a966239ca', '54238fd6-d2d0-4408-b1a9-d19d24fd29ce', 'f140a2ec-fd49-4814-994a-fe3476f14e66', '73918ae1-e4fd-4c18-b132-00cb555b1ad2', '6f09ba7e-e3ce-44b0-932b-c003fb44fb89', '7cb81727-2097-4b52-b480-c89867b5b34c', 'db4df448-e449-4a6f-a0e7-288711e7a75a', 'ca4ecb4c-4b60-4723-9b9e-2c54a6290a53', 'b22f694e-4a34-4142-ab9d-2556c3487086', 'f115196e-8dfe-4d2a-8af3-8206d93c1729', 'ff96bfe1-d925-4553-94b5-bf8297adf259', 'b03fbc44-3d8e-4a6c-8a50-5ea3498568e0', '3638d102-e8b6-4230-8742-e548cd87a949']
['dc962048-89bb-4e6a-96a9-b062a2be1426', '1b715600-0cbc-442c-bd00-5b0ac2865de1', '8928f98a-b411-497e-aa4b-aa752434686d', 'd0ea3148-948d-4817-94f8-dcaf2342bbbe', 'e535fb62-e245-4a48-b119-88ce62a6fe67', '3bcb81b4-d9ca-4fc9-a1cd-353a9662

In [18]:
! python /home/bsb2144/daart/examples/fit_models_subsample_loop.py --pre wv2 --fit_mlp --frac 1 --dataset ibl --n_samples 1 --data /home/bsb2144/daart_utils/configs/data_ibl3.yaml --model /home/bsb2144/daart_utils/configs/model_ibl4.yaml --train /home/bsb2144/daart_utils/configs/train_ibl4.yaml

here
mod types ['temporal-mlp']
frac 1
n_sessions 18
['dc962048-89bb-4e6a-96a9-b062a2be1426', '1b715600-0cbc-442c-bd00-5b0ac2865de1', '8928f98a-b411-497e-aa4b-aa752434686d', 'd0ea3148-948d-4817-94f8-dcaf2342bbbe', 'e535fb62-e245-4a48-b119-88ce62a6fe67', '3bcb81b4-d9ca-4fc9-a1cd-353a966239ca', '54238fd6-d2d0-4408-b1a9-d19d24fd29ce', 'f140a2ec-fd49-4814-994a-fe3476f14e66', '73918ae1-e4fd-4c18-b132-00cb555b1ad2', '6f09ba7e-e3ce-44b0-932b-c003fb44fb89', '7cb81727-2097-4b52-b480-c89867b5b34c', 'db4df448-e449-4a6f-a0e7-288711e7a75a', 'ca4ecb4c-4b60-4723-9b9e-2c54a6290a53', 'b22f694e-4a34-4142-ab9d-2556c3487086', 'f115196e-8dfe-4d2a-8af3-8206d93c1729', 'ff96bfe1-d925-4553-94b5-bf8297adf259', 'b03fbc44-3d8e-4a6c-8a50-5ea3498568e0', '3638d102-e8b6-4230-8742-e548cd87a949']
['dc962048-89bb-4e6a-96a9-b062a2be1426', '1b715600-0cbc-442c-bd00-5b0ac2865de1', '8928f98a-b411-497e-aa4b-aa752434686d', 'd0ea3148-948d-4817-94f8-dcaf2342bbbe', 'e535fb62-e245-4a48-b119-88ce62a6fe67', '3bcb81b4-d9ca-4fc9-a1cd-